# 04 - Quantitative Evaluation

This notebook evaluates the frozen final extractions. Technical completion is reported for all nine documents, while the primary extraction-quality comparison uses only the five documents completed by both workflows. No predictions are reconstructed from partial chunk outputs.

## 1. Setup and frozen inputs

In [1]:
# Import only local utilities needed for deterministic evaluation.
from pathlib import Path
import json
import math
import re
import sys
import unicodedata

import pandas as pd

# Locate the repository whether the notebook starts from the root or notebooks/.
PROJECT_FOLDER = Path.cwd().resolve()
for candidate in [PROJECT_FOLDER, PROJECT_FOLDER.parent]:
    if (candidate / "data/evaluation/manual_annotations.csv").exists():
        PROJECT_FOLDER = candidate
        break
else:
    raise FileNotFoundError("Could not find the project data directory")

# Reuse the same Pydantic output schema used during extraction.
sys.path.insert(0, str(PROJECT_FOLDER / "src"))
from extraction_schema import DocumentExtraction

# Keep references, predictions, source documents, and audit output separate.
REFERENCE_PATH = PROJECT_FOLDER / "data/evaluation/manual_annotations.csv"
MANIFEST_PATH = PROJECT_FOLDER / "data/evaluation/manifest.json"
DOCUMENT_DIR = PROJECT_FOLDER / "data/evaluation/documents"
RESULTS_DIR = PROJECT_FOLDER / "data/final_results"
SUMMARY_PATH = RESULTS_DIR / "experiment_summary.csv"
UNSUPPORTED_EVIDENCE_PATH = PROJECT_FOLDER / "outputs/unsupported_evidence_cases.csv"

print(f"Project folder: {PROJECT_FOLDER}")

Project folder: C:\Users\onssr\Desktop\AIHWS\text2hazard_project


## 2. Load and validate the reference and predictions

In [2]:
# Load the frozen manual reference annotations and experiment summary.
reference = pd.read_csv(REFERENCE_PATH).fillna("")
summary = pd.read_csv(SUMMARY_PATH)

reference_counts = pd.DataFrame({
    "measure": ["documents", "CSV rows", "annotated facts", "no-event rows"],
    "count": [
        reference["document_id"].nunique(),
        len(reference),
        (reference["fact_type"] != "no_event").sum(),
        (reference["fact_type"] == "no_event").sum(),
    ],
})
display(reference_counts)

fact_counts = (
    reference.loc[reference["fact_type"] != "no_event", "fact_type"]
    .value_counts()
    .reindex(["hazard", "location", "time", "impact"], fill_value=0)
    .rename_axis("fact_type")
    .reset_index(name="reference_facts")
)
display(fact_counts)

# These assertions prevent accidental evaluation with a changed reference file.
assert len(reference) == 110
assert reference["document_id"].nunique() == 9
assert (reference["fact_type"] != "no_event").sum() == 108
assert (reference["fact_type"] == "no_event").sum() == 2
assert fact_counts.set_index("fact_type")["reference_facts"].to_dict() == {
    "hazard": 14, "location": 25, "time": 6, "impact": 63
}
assert set(reference["document_id"]) == set(summary["document_id"])


# A missing final.json represents a technical failure, not an empty prediction.
def load_final_output(workflow, document_number, expected_document_id):
    path = RESULTS_DIR / workflow / f"document_{document_number:02d}" / "final.json"
    if not path.exists():
        return None
    output = DocumentExtraction.model_validate_json(path.read_text(encoding="utf-8"))
    assert output.document_id == expected_document_id
    return output.model_dump(mode="json")


# Load only validated document-level outputs and record technical completion.
predictions = {"Workflow A": {}, "Workflow B": {}}
completion_rows = []
for index, row in summary.iterrows():
    number = index + 1
    document_id = row["document_id"]
    output_a = load_final_output("workflow_a", number, document_id)
    output_b = load_final_output("workflow_b", number, document_id)
    predictions["Workflow A"][document_id] = output_a
    predictions["Workflow B"][document_id] = output_b
    completion_rows.append({
        "document_id": document_id,
        "workflow_a_complete": output_a is not None,
        "workflow_b_complete": output_b is not None,
    })

completion = pd.DataFrame(completion_rows)
display(completion)
assert completion["workflow_a_complete"].sum() == 5
assert completion["workflow_b_complete"].sum() == 7

,measure,count
0,documents,9
1,CSV rows,110
2,annotated facts,108
3,no-event rows,2


,fact_type,reference_facts
0,hazard,14
1,location,25
2,time,6
3,impact,63


,document_id,workflow_a_complete,workflow_b_complete
0,PWC 2015 - Updated estimates on cost of Storm ...,True,True
1,Krausmann 2014 - STREST report on lessons lear...,True,True
2,AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLU...,False,True
3,Lavenir 2021 - À la recherche d'une jeune fill...,True,True
4,ABC 2024 - Traffic jams and flight delays due ...,True,True
5,"EFE 2024 - The DANA storm, live_ The death tol...",False,True
6,Plag 2014 - Foreword extreme geohazards—a grow...,True,True
7,Kadir 2014 - The Impact of Natural Disasters o...,False,False
8,"Cavaleri 2019 - The October 29, 2018 storm in ...",False,False


In [3]:
# Reconstruct the reference event structure from the row-based annotation CSV.
FACT_TYPES = ["hazard", "location", "time", "impact"]

reference_events = {}
no_event_documents = set()
for document_id, document_rows in reference.groupby("document_id", sort=False):
    no_event_rows = document_rows[document_rows["fact_type"] == "no_event"]
    if not no_event_rows.empty:
        no_event_documents.add(document_id)
        reference_events[document_id] = []
        continue

    events = []
    for event_id, event_rows in document_rows.groupby("event_id", sort=False):
        event = {"event_id": event_id, **{fact_type: [] for fact_type in FACT_TYPES}}
        for fact_type in FACT_TYPES:
            event[fact_type] = event_rows[event_rows["fact_type"] == fact_type].to_dict("records")
        events.append(event)
    reference_events[document_id] = events

assert no_event_documents == {
    "Lavenir 2021 - À la recherche d'une jeune fille tombée dans l'Ourthe à Marcourt - L'Avenir",
    "Plag 2014 - Foreword extreme geohazards—a growing threat for a globally interconnected civilization",
}
assert all(len(events) <= 1 for events in reference_events.values())

# The paired comparison uses only documents completed by both workflows.
PAIRED_DOCUMENTS = completion.loc[
    completion["workflow_a_complete"] & completion["workflow_b_complete"],
    "document_id",
].tolist()
# AEMET and EFE are evaluated separately because only Workflow B completed them.
B_ONLY_DOCUMENTS = completion.loc[
    ~completion["workflow_a_complete"] & completion["workflow_b_complete"],
    "document_id",
].tolist()

print("Paired evaluation subset:")
for document_id in PAIRED_DOCUMENTS:
    print(f"- {document_id}")
print("\nAdditional Workflow B subset:")
for document_id in B_ONLY_DOCUMENTS:
    print(f"- {document_id}")

assert len(PAIRED_DOCUMENTS) == 5
assert {document_id.split(" ", 1)[0] for document_id in B_ONLY_DOCUMENTS} == {"AEMET", "EFE"}

Paired evaluation subset:
- PWC 2015 - Updated estimates on cost of Storm Desmond
- Krausmann 2014 - STREST report on lessons learned from recent catastrophic events
- Lavenir 2021 - À la recherche d'une jeune fille tombée dans l'Ourthe à Marcourt - L'Avenir
- ABC 2024 - Traffic jams and flight delays due to heavy rain and lightning storm in Malaga
- Plag 2014 - Foreword extreme geohazards—a growing threat for a globally interconnected civilization

Additional Workflow B subset:
- AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLUVIAS INTENSAS
- EFE 2024 - The DANA storm, live_ The death toll rises to 158


## 3. Technical robustness and efficiency

These measures describe whether the frozen API run produced complete document-level outputs. They are not extraction-quality scores. Failure counts come from the saved experiment summary; there are no reconstructed outputs or retries.

In [4]:
# Count failure messages without mixing technical reliability with fact quality.
def count_failure(summary_frame, column, phrase):
    return int(summary_frame[column].fillna("").str.count(phrase).sum())


# Summarize completion, calls, runtime, truncation, parsing failures, and timeouts.
technical_rows = []
for workflow, prefix in [("Workflow A", "workflow_a"), ("Workflow B", "workflow_b")]:
    completed = int((summary[f"{prefix}_status"] == "ok").sum())
    failure_column = f"{prefix}_failure_reason"
    technical_rows.append({
        "workflow": workflow,
        "completed_documents": completed,
        "failed_documents": len(summary) - completed,
        "completion_rate": completed / len(summary),
        "logical_api_calls": int(summary[f"{prefix}_logical_calls"].sum()),
        "total_runtime_seconds": summary[f"{prefix}_duration_seconds"].sum(),
        "truncation_failures": count_failure(summary, failure_column, "truncated at the output-token limit"),
        "invalid_structured_responses": count_failure(summary, failure_column, "no valid structured response"),
        "timeouts": count_failure(summary, failure_column, r"(?i)timeout|timed out"),
    })

technical_robustness = pd.DataFrame(technical_rows)
technical_display = technical_robustness.copy()
technical_display["completion_rate"] = technical_display["completion_rate"].round(3)
technical_display["total_runtime_seconds"] = technical_display["total_runtime_seconds"].round(3)
display(technical_display)

,workflow,completed_documents,failed_documents,completion_rate,logical_api_calls,total_runtime_seconds,truncation_failures,invalid_structured_responses,timeouts
0,Workflow A,5,4,0.556,52,2117.296,3,6,0
1,Workflow B,7,2,0.778,202,2861.814,0,2,0


## 4. Conservative matching rules

- **Primary unit:** facts are matched one-to-one across the whole document, independently of predicted event IDs. Duplicate and unsupported predictions remain false positives. This corresponds directly to the research question about hazard-information extraction; requiring a correct fact to occur in one aligned predicted event would mix extraction quality with event-structure reconstruction.
- **Hazards:** normalized hazard category is the comparison target. Source wording does not reject an otherwise correct normalized category. One-to-one matching means repeated category predictions remain false positives.
- **Locations:** normalized names are compared when both sides provide them, and country codes must agree when both exist. Conservative normalized source-text matching is used only when a normalized name is unavailable.
- **Times:** normalized start/end values are used when available. Otherwise normalized source expressions are compared. A written day-month-year expression is converted only for matching against an existing normalized reference date.
- **Impacts:** categories must agree. Numbers, currencies/units, and polarity or uncertainty markers must not conflict. Remaining content tokens require at least two shared tokens and Jaccard similarity of at least 0.45. A small fixed word map handles forms such as *died/deaths*, *injured/injuries*, and *evacuated/evacuation*.

Evidence quotations are not used to decide matches. Matching is lexical and deterministic; no LLM, embedding model, or external semantic evaluator is used. The deterministic greedy one-to-one matcher was checked against maximum-cardinality matching for all 48 evaluated document/workflow/field combinations, and no TP, FP, FN, or F1 value changed. This empirical check does not imply that greedy matching guarantees a globally maximum assignment in general.

In [5]:
# Small transparent vocabularies support conservative lexical normalization.
STOPWORDS = {
    "a", "an", "the", "of", "in", "on", "at", "to", "for", "with", "and", "or",
    "by", "from", "as", "is", "are", "was", "were", "be", "been", "being", "that",
    "this", "these", "those", "its", "their", "his", "her", "into", "after", "due",
}

WORD_MAP = {
    "died": "death", "dead": "death", "deaths": "death", "fatality": "death", "fatalities": "death",
    "injured": "injury", "injuries": "injury",
    "evacuated": "evacuate", "evacuation": "evacuate", "evacuations": "evacuate",
    "rescued": "rescue", "rescues": "rescue",
    "collapsed": "collapse", "collapses": "collapse",
    "destroyed": "destroy", "destruction": "destroy",
    "damaged": "damage", "damages": "damage",
    "costs": "cost", "payouts": "payout", "payments": "payment", "paying": "payout",
    "insurer": "insurance", "insurers": "insurance",
    "floods": "flood", "flooding": "flood", "storms": "storm",
    "closures": "closure", "closed": "closure", "delays": "delay", "delayed": "delay",
    "buildings": "building", "roads": "road", "residents": "resident",
    "persons": "person", "people": "person", "families": "family",
}

LOCATION_QUALIFIERS = {"province", "city", "airport", "river", "region", "prefecture", "country"}
POLARITY_MARKERS = {
    "no", "not", "without", "potential", "potentially", "possible", "possibly", "risk",
    "could", "may", "might", "would", "expected", "forecast", "projected",
}
QUANTIFIER_MARKERS = {"about", "approximately", "over", "between", "least", "more"}
MONTHS = {
    "january": 1, "february": 2, "march": 3, "april": 4, "may": 5, "june": 6,
    "july": 7, "august": 8, "september": 9, "october": 10, "november": 11, "december": 12,
}


# Normalize case, accents, whitespace, and dash variants before comparison.
def normalized_string(value):
    value = unicodedata.normalize("NFKD", str(value or "")).casefold()
    value = "".join(character for character in value if not unicodedata.combining(character))
    value = value.replace("‑", "-").replace("–", "-").replace("—", "-")
    return " ".join(value.split())


def tokens(value):
    raw = re.findall(r"[a-z]+|\d+(?:[.,]\d+)*", normalized_string(value))
    return {
        WORD_MAP.get(token, token)
        for token in raw
        if token not in STOPWORDS
    }


def overlap_coefficient(left, right):
    if not left or not right:
        return 0.0
    return len(left & right) / min(len(left), len(right))


def jaccard(left, right):
    if not left or not right:
        return 0.0
    return len(left & right) / len(left | right)


# Preserve numbers, units, currencies, polarity, and qualifiers for impact matching.
def number_signature(value):
    values = re.findall(r"\d+(?:[.,]\d+)*", normalized_string(value))
    return tuple(sorted(number.replace(",", "") for number in values))


def currency_unit_signature(value):
    text = normalized_string(value)
    signatures = set(re.findall(r"[£€¥$%]|(?:usd|eur|gbp|dollar|dollars|euro|euros|yen|billion|million|thousand|km|mm)", text))
    return signatures


def marker_signature(value):
    word_set = tokens(value)
    return (word_set & POLARITY_MARKERS, word_set & QUANTIFIER_MARKERS)


def written_date(value):
    text = normalized_string(value)
    match = re.search(r"(\d{1,2})\s+([a-z]+)\s+(\d{4})", text)
    if not match or match.group(2) not in MONTHS:
        return None
    return f"{int(match.group(3)):04d}-{MONTHS[match.group(2)]:02d}-{int(match.group(1)):02d}"


# Map the singular evaluation field name to its list in the extraction schema.
def prediction_facts(event, fact_type):
    return event[{"hazard": "hazards", "location": "locations", "time": "times", "impact": "impacts"}[fact_type]]

In [6]:
# Hazards match when their normalized schema categories are identical.
def hazard_score(reference_fact, predicted_fact):
    if reference_fact["category"] != predicted_fact["category"]:
        return 0.0
    return 1.0


# Prefer normalized location names and compatible country codes; use a
# conservative source-text fallback only when normalized names are unavailable.
def location_score(reference_fact, predicted_fact):
    ref_country = reference_fact["country_code"]
    pred_country = predicted_fact.get("country_code") or ""
    if ref_country and pred_country and ref_country != pred_country:
        return 0.0

    ref_name = reference_fact["normalized_name"]
    pred_name = predicted_fact.get("normalized_name") or ""
    if ref_name and pred_name:
        return 3.0 if normalized_string(ref_name) == normalized_string(pred_name) else 0.0

    left = tokens(reference_fact["source_text"])
    right = tokens(predicted_fact["source_text"])
    if (left & LOCATION_QUALIFIERS) != (right & LOCATION_QUALIFIERS):
        return 0.0
    overlap = overlap_coefficient(left, right)
    return 1.0 + overlap if overlap >= 0.8 else 0.0


# Prefer normalized start/end values and retain conservative text comparison
# for unresolved relative expressions.
def time_score(reference_fact, predicted_fact):
    ref_start = reference_fact["normalized_start"]
    ref_end = reference_fact["normalized_end"]
    pred_start = predicted_fact.get("normalized_start") or ""
    pred_end = predicted_fact.get("normalized_end") or ""

    if ref_start and pred_start:
        if ref_start == pred_start and (not ref_end or not pred_end or ref_end == pred_end):
            return 2.0
        return 0.0
    if ref_start and written_date(predicted_fact["source_text"]) == ref_start:
        return 1.9

    left = tokens(reference_fact["source_text"])
    right = tokens(predicted_fact["source_text"])
    overlap = overlap_coefficient(left, right)
    return 1.0 + overlap if overlap >= 0.8 else 0.0


# Impact matching remains deliberately strict about category, numbers, units,
# qualifiers, polarity, and lexical overlap.
def impact_score(reference_fact, predicted_fact):
    if reference_fact["category"] != predicted_fact["category"]:
        return 0.0
    reference_text = reference_fact["description"]
    predicted_text = predicted_fact["description"]

    if number_signature(reference_text) != number_signature(predicted_text):
        return 0.0
    if currency_unit_signature(reference_text) != currency_unit_signature(predicted_text):
        return 0.0
    if marker_signature(reference_text) != marker_signature(predicted_text):
        return 0.0

    left = tokens(reference_text)
    right = tokens(predicted_text)
    shared = left & right
    similarity = jaccard(left, right)
    if normalized_string(reference_text) == normalized_string(predicted_text):
        return 3.0
    if len(shared) >= 2 and similarity >= 0.45:
        return 1.0 + similarity
    return 0.0


SCORE_FUNCTIONS = {
    "hazard": hazard_score,
    "location": location_score,
    "time": time_score,
    "impact": impact_score,
}


# Match facts one-to-one. Extra or repeated predictions remain false positives.
def match_facts(reference_facts, predicted_facts, fact_type):
    candidates = []
    scorer = SCORE_FUNCTIONS[fact_type]
    for reference_index, reference_fact in enumerate(reference_facts):
        for predicted_index, predicted_fact in enumerate(predicted_facts):
            score = scorer(reference_fact, predicted_fact)
            if score > 0:
                candidates.append((score, reference_index, predicted_index))

    matched_reference = set()
    matched_predictions = set()
    matches = []
    for score, reference_index, predicted_index in sorted(
        candidates, key=lambda item: (-item[0], item[1], item[2])
    ):
        if reference_index in matched_reference or predicted_index in matched_predictions:
            continue
        matched_reference.add(reference_index)
        matched_predictions.add(predicted_index)
        matches.append((reference_index, predicted_index, score))
    return matches


# Event alignment is calculated separately for the structural fragmentation audit.
def event_pair_score(reference_event, predicted_event):
    score = 0
    for fact_type in FACT_TYPES:
        score += len(match_facts(
            reference_event[fact_type], prediction_facts(predicted_event, fact_type), fact_type
        ))
    return score


# Align events deterministically for structural reporting, not primary fact scoring.
def align_events(reference_event_list, predicted_event_list):
    candidates = []
    for reference_index, reference_event in enumerate(reference_event_list):
        for predicted_index, predicted_event in enumerate(predicted_event_list):
            score = event_pair_score(reference_event, predicted_event)
            if score > 0:
                candidates.append((score, reference_index, predicted_index))

    used_reference = set()
    used_predictions = set()
    alignment = []
    for score, reference_index, predicted_index in sorted(
        candidates, key=lambda item: (-item[0], item[1], item[2])
    ):
        if reference_index in used_reference or predicted_index in used_predictions:
            continue
        used_reference.add(reference_index)
        used_predictions.add(predicted_index)
        alignment.append((reference_index, predicted_index, score))
    return alignment

In [7]:
# Return undefined metrics as NaN instead of forcing an artificial zero.
def safe_divide(numerator, denominator):
    return numerator / denominator if denominator else math.nan


def metric_values(tp, fp, fn):
    precision = safe_divide(tp, tp + fp)
    recall = safe_divide(tp, tp + fn)
    if math.isnan(precision) or math.isnan(recall):
        f1 = math.nan
    elif precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1


# Primary evaluation pools each fact type across the whole document. Event IDs
# do not restrict fact matching, so extraction quality and structure stay separate.
def evaluate_document(document_id, prediction):
    reference_event_list = reference_events[document_id]
    predicted_event_list = prediction["events"]
    alignment = align_events(reference_event_list, predicted_event_list)
    rows = []

    for fact_type in FACT_TYPES:
        reference_facts = [
            fact for event in reference_event_list for fact in event[fact_type]
        ]
        predicted_facts = [
            fact
            for event in predicted_event_list
            for fact in prediction_facts(event, fact_type)
        ]
        true_positives = len(match_facts(reference_facts, predicted_facts, fact_type))
        reference_count = len(reference_facts)
        prediction_count = len(predicted_facts)
        false_positives = prediction_count - true_positives
        false_negatives = reference_count - true_positives
        precision, recall, f1 = metric_values(
            true_positives, false_positives, false_negatives
        )
        rows.append({
            "document_id": document_id,
            "fact_type": fact_type,
            "tp": true_positives,
            "fp": false_positives,
            "fn": false_negatives,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        })
    return rows, alignment


# Evaluate only successfully completed documents in the requested quality subset.
def evaluate_subset(workflow, document_ids):
    rows = []
    alignments = {}
    for document_id in document_ids:
        document_rows, alignment = evaluate_document(
            document_id, predictions[workflow][document_id]
        )
        rows.extend(document_rows)
        alignments[document_id] = alignment
    return pd.DataFrame(rows), alignments


# Sum TP, FP, and FN before calculating micro precision, recall, and F1.
def aggregate_metrics(result_rows, group_columns):
    grouped = result_rows.groupby(group_columns, as_index=False)[["tp", "fp", "fn"]].sum()
    values = grouped.apply(
        lambda row: metric_values(row["tp"], row["fp"], row["fn"]), axis=1
    )
    grouped[["precision", "recall", "f1"]] = pd.DataFrame(values.tolist(), index=grouped.index)
    return grouped

## 5. Primary paired extraction-quality comparison

The primary metrics use deterministic one-to-one fact matching across each entire document. Predicted event IDs do not gate matches, while duplicate and unsupported predictions still count as false positives.

In [8]:
# Compare A and B fairly on the five documents completed by both workflows.
paired_frames = []
paired_alignments = {}
for workflow in ["Workflow A", "Workflow B"]:
    results, alignments = evaluate_subset(workflow, PAIRED_DOCUMENTS)
    results.insert(0, "workflow", workflow)
    paired_frames.append(results)
    paired_alignments[workflow] = alignments

paired_results = pd.concat(paired_frames, ignore_index=True)
paired_per_field = aggregate_metrics(paired_results, ["workflow", "fact_type"])
paired_per_field["fact_type"] = pd.Categorical(
    paired_per_field["fact_type"], FACT_TYPES, ordered=True
)
paired_per_field = paired_per_field.sort_values(["workflow", "fact_type"])

paired_overall = aggregate_metrics(paired_results, ["workflow"])

print("Per-field micro metrics on the five paired documents:")
display(paired_per_field.round({"precision": 3, "recall": 3, "f1": 3}))
print("Overall micro metrics across all four fact types:")
display(paired_overall.round({"precision": 3, "recall": 3, "f1": 3}))

Per-field micro metrics on the five paired documents:


,workflow,fact_type,tp,fp,fn,precision,recall,f1
0,Workflow A,hazard,4,58,2,0.065,0.667,0.118
2,Workflow A,location,4,22,1,0.154,0.800,0.258
3,Workflow A,time,2,20,0,0.091,1.000,0.167
1,Workflow A,impact,11,21,11,0.344,0.500,0.407
4,Workflow B,hazard,3,75,3,0.038,0.500,0.071
6,Workflow B,location,2,11,3,0.154,0.400,0.222
7,Workflow B,time,2,2,0,0.500,1.000,0.667
5,Workflow B,impact,10,32,12,0.238,0.455,0.312


Overall micro metrics across all four fact types:


,workflow,tp,fp,fn,precision,recall,f1
0,Workflow A,21,121,14,0.148,0.600,0.237
1,Workflow B,17,120,18,0.124,0.486,0.198


In [9]:
# Keep document-level results visible alongside the pooled paired metrics.
# TP, FP, and FN are summed across hazard, location, time, and impact facts
# separately for each workflow-document pair. This shows whether the overall
# result is consistent or driven by particularly easy or difficult documents.
paired_per_document = aggregate_metrics(
    paired_results, ["workflow", "document_id"]
)
# Rounding affects only the displayed table, not the underlying metric values.
display(paired_per_document.round({"precision": 3, "recall": 3, "f1": 3}))

,workflow,document_id,tp,fp,fn,precision,recall,f1
0,Workflow A,ABC 2024 - Traffic jams and flight delays due ...,7,22,4,0.241,0.636,0.350
1,Workflow A,Krausmann 2014 - STREST report on lessons lear...,11,82,9,0.118,0.550,0.195
2,Workflow A,Lavenir 2021 - À la recherche d'une jeune fill...,0,7,0,0.000,NaN,NaN
3,Workflow A,PWC 2015 - Updated estimates on cost of Storm ...,3,10,1,0.231,0.750,0.353
4,Workflow A,Plag 2014 - Foreword extreme geohazards—a grow...,0,0,0,NaN,NaN,NaN
5,Workflow B,ABC 2024 - Traffic jams and flight delays due ...,3,64,8,0.045,0.273,0.077
6,Workflow B,Krausmann 2014 - STREST report on lessons lear...,10,39,10,0.204,0.500,0.290
7,Workflow B,Lavenir 2021 - À la recherche d'une jeune fill...,0,0,0,NaN,NaN,NaN
8,Workflow B,PWC 2015 - Updated estimates on cost of Storm ...,4,6,0,0.400,1.000,0.571
9,Workflow B,Plag 2014 - Foreword extreme geohazards—a grow...,0,11,0,0.000,NaN,NaN


## 6. Secondary event-structure evaluation

Event structure is reported separately from document-level fact quality. Event IDs are bookkeeping labels, not semantic ground-truth identifiers. Events are aligned deterministically by their matched facts only to measure fragmentation; unaligned-event facts do not affect the primary document-level matching beyond their ordinary treatment as predicted facts.

In [10]:
# Quantify event fragmentation separately from document-level fact quality.
fragmentation_rows = []
for workflow, completion_column in [
    ("Workflow A", "workflow_a_complete"),
    ("Workflow B", "workflow_b_complete"),
]:
    completed_documents = completion.loc[completion[completion_column], "document_id"]
    for document_id in completed_documents:
        reference_event_list = reference_events[document_id]
        predicted_event_list = predictions[workflow][document_id]["events"]
        alignment = align_events(reference_event_list, predicted_event_list)
        aligned_prediction_indices = {predicted_index for _, predicted_index, _ in alignment}
        unaligned_prediction_indices = set(range(len(predicted_event_list))) - aligned_prediction_indices
        facts_in_unaligned_events = sum(
            len(prediction_facts(predicted_event_list[index], fact_type))
            for index in unaligned_prediction_indices
            for fact_type in FACT_TYPES
        )
        fragmentation_rows.append({
            "document_id": document_id,
            "workflow": workflow,
            "reference_events": len(reference_event_list),
            "predicted_events": len(predicted_event_list),
            "excess_predicted_events": max(
                len(predicted_event_list) - len(reference_event_list), 0
            ),
            "unaligned_predicted_events": len(unaligned_prediction_indices),
            "facts_in_unaligned_events": facts_in_unaligned_events,
        })

event_fragmentation = pd.DataFrame(fragmentation_rows)
display(event_fragmentation)

,document_id,workflow,reference_events,predicted_events,excess_predicted_events,unaligned_predicted_events,facts_in_unaligned_events
0,PWC 2015 - Updated estimates on cost of Storm ...,Workflow A,1,2,1,1,4
1,Krausmann 2014 - STREST report on lessons lear...,Workflow A,1,8,7,7,40
2,Lavenir 2021 - À la recherche d'une jeune fill...,Workflow A,0,1,1,1,7
3,ABC 2024 - Traffic jams and flight delays due ...,Workflow A,1,7,6,6,25
4,Plag 2014 - Foreword extreme geohazards—a grow...,Workflow A,0,0,0,0,0
5,PWC 2015 - Updated estimates on cost of Storm ...,Workflow B,1,1,0,0,0
6,Krausmann 2014 - STREST report on lessons lear...,Workflow B,1,7,6,6,39
7,AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLU...,Workflow B,1,63,62,62,413
8,Lavenir 2021 - À la recherche d'une jeune fill...,Workflow B,0,0,0,0,0
9,ABC 2024 - Traffic jams and flight delays due ...,Workflow B,1,20,19,19,42


## 7. No-event documents

A no-event document is correct only when the reference is `no_event` and the prediction contains no events. Fact-level precision, recall, and F1 remain undefined when both sides contain zero facts; no artificial perfect score is assigned.

In [11]:
# Check whether each workflow correctly returns no events for no-event documents.
no_event_rows = []
for document_id in sorted(no_event_documents):
    for workflow in ["Workflow A", "Workflow B"]:
        prediction = predictions[workflow][document_id]
        predicted_events = len(prediction["events"])
        predicted_facts = sum(
            len(prediction_facts(event, fact_type))
            for event in prediction["events"]
            for fact_type in FACT_TYPES
        )
        no_event_rows.append({
            "document_id": document_id,
            "workflow": workflow,
            "predicted_events": predicted_events,
            "predicted_facts": predicted_facts,
            "correct_no_event": predicted_events == 0,
        })

no_event_results = pd.DataFrame(no_event_rows)
display(no_event_results)

,document_id,workflow,predicted_events,predicted_facts,correct_no_event
0,Lavenir 2021 - À la recherche d'une jeune fill...,Workflow A,1,7,False
1,Lavenir 2021 - À la recherche d'une jeune fill...,Workflow B,0,0,True
2,Plag 2014 - Foreword extreme geohazards—a grow...,Workflow A,0,0,True
3,Plag 2014 - Foreword extreme geohazards—a grow...,Workflow B,5,11,False


## 8. Additional Workflow B results: AEMET and EFE

These two documents are evaluated separately because Workflow A did not produce complete document-level outputs. They are not included in the primary paired comparison.

In [12]:
# Report AEMET and EFE as additional Workflow B results, not as paired evidence.
b_only_results, b_only_alignments = evaluate_subset(
    "Workflow B", B_ONLY_DOCUMENTS
)
b_only_per_field = aggregate_metrics(b_only_results, ["fact_type"])
b_only_per_field["fact_type"] = pd.Categorical(
    b_only_per_field["fact_type"], FACT_TYPES, ordered=True
)
b_only_per_field = b_only_per_field.sort_values("fact_type")
b_only_overall = aggregate_metrics(
    b_only_results.assign(subset="Workflow B: AEMET + EFE"), ["subset"]
)
b_only_per_document = aggregate_metrics(b_only_results, ["document_id"])

print("Per-field micro metrics:")
display(b_only_per_field.round({"precision": 3, "recall": 3, "f1": 3}))
print("Overall micro metrics:")
display(b_only_overall.round({"precision": 3, "recall": 3, "f1": 3}))
print("Per-document metrics:")
display(b_only_per_document.round({"precision": 3, "recall": 3, "f1": 3}))

Per-field micro metrics:


,fact_type,tp,fp,fn,precision,recall,f1
0,hazard,4,300,0,0.013,1.000,0.026
2,location,12,221,2,0.052,0.857,0.097
3,time,1,92,1,0.011,0.500,0.021
1,impact,3,53,29,0.054,0.094,0.068


Overall micro metrics:


,subset,tp,fp,fn,precision,recall,f1
0,Workflow B: AEMET + EFE,20,666,32,0.029,0.385,0.054


Per-document metrics:


,document_id,tp,fp,fn,precision,recall,f1
0,AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLU...,15,405,22,0.036,0.405,0.066
1,"EFE 2024 - The DANA storm, live_ The death tol...",5,261,10,0.019,0.333,0.036


## 9. Evidence validity

Evidence support is checked independently of fact matching. A predicted quotation is supported only when its exact character sequence occurs in the corresponding cleaned Markdown; quotations are not normalized, repaired, or fuzzy-matched.

In [ ]:
# Reload the cleaned Markdown only to verify evidence quotations verbatim.
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
document_texts = {
    item["document_id"]: (DOCUMENT_DIR / item["filename"]).read_text(encoding="utf-8")
    for item in manifest["documents"]
}


# Evidence is supported only when the exact quoted characters occur in the source.
def evidence_rows(workflow, document_ids):
    rows = []
    for document_id in document_ids:
        document_text = document_texts[document_id]
        for event in predictions[workflow][document_id]["events"]:
            for fact_type in FACT_TYPES:
                for fact in prediction_facts(event, fact_type):
                    evidence_quote = fact["evidence"]["quote"]
                    rows.append({
                        "document_id": document_id,
                        "workflow": workflow,
                        "event_id": event["event_id"],
                        "fact_type": fact_type,
                        "evidence_quote": evidence_quote,
                        "supported": evidence_quote in document_text,
                    })
    return pd.DataFrame(rows)


# Summarize provenance support separately from semantic extraction correctness.
def evidence_summary(rows, subset_name):
    summaries = []
    for workflow, workflow_rows in rows.groupby("workflow", sort=False):
        total = len(workflow_rows)
        supported = int(workflow_rows["supported"].sum())
        unsupported = total - supported
        summaries.append({
            "subset": subset_name,
            "workflow": workflow,
            "total_predicted_facts": total,
            "supported_evidence_quotes": supported,
            "unsupported_evidence_quotes": unsupported,
            "supported_evidence_rate": safe_divide(supported, total),
            "unsupported_evidence_rate": safe_divide(unsupported, total),
        })
    return pd.DataFrame(summaries)


# Collect evidence checks for the five documents completed by both workflows.
# Comparison between Workflow A and Workflow B.
paired_evidence = pd.concat([
    evidence_rows("Workflow A", PAIRED_DOCUMENTS),
    evidence_rows("Workflow B", PAIRED_DOCUMENTS),
], ignore_index=True)

# Collect evidence checks from every technically complete output.
# Workflow A and Workflow B can therefore include different document sets here.
all_complete_evidence = pd.concat([
    evidence_rows(
        workflow,
        completion.loc[completion[completion_column], "document_id"].tolist(),
    )
    for workflow, completion_column in [
        ("Workflow A", "workflow_a_complete"),
        ("Workflow B", "workflow_b_complete"),
    ]
], ignore_index=True)

# Calculate evidence-support rates for the paired comparison and for all technically complete outputs. 
# Evidence is supported only when its quotation occurs verbatim in the corresponding cleaned Markdown document.
evidence_validity = pd.concat([
    evidence_summary(paired_evidence, "Five paired documents"),
    evidence_summary(all_complete_evidence, "All complete outputs"),
], ignore_index=True)

# Rounding is used only for displaying the table.
display(evidence_validity.round({
    "supported_evidence_rate": 3,
    "unsupported_evidence_rate": 3,
}))

# Keep unsupported quotations for transparent manual inspection.
unsupported_evidence_cases = all_complete_evidence.loc[
    ~all_complete_evidence["supported"],
    ["document_id", "workflow", "event_id", "fact_type", "evidence_quote"],
].reset_index(drop=True)

# Save the audit cases separately from the frozen extraction results.
unsupported_evidence_cases.to_csv(
    UNSUPPORTED_EVIDENCE_PATH, index=False, encoding="utf-8"
)

display(unsupported_evidence_cases)
print(f"Saved unsupported evidence cases to: {UNSUPPORTED_EVIDENCE_PATH}")

,subset,workflow,total_predicted_facts,supported_evidence_quotes,unsupported_evidence_quotes,supported_evidence_rate,unsupported_evidence_rate
0,Five paired documents,Workflow A,142,138,4,0.972,0.028
1,Five paired documents,Workflow B,137,131,6,0.956,0.044
2,All complete outputs,Workflow A,142,138,4,0.972,0.028
3,All complete outputs,Workflow B,823,772,51,0.938,0.062


,document_id,workflow,event_id,fact_type,evidence_quote
0,Krausmann 2014 - STREST report on lessons lear...,Workflow A,event_01,hazard,the tsunami it triggered...
1,Krausmann 2014 - STREST report on lessons lear...,Workflow A,event_08,impact,The explosions damaged nearby asphalt tanks du...
2,Lavenir 2021 - À la recherche d'une jeune fill...,Workflow A,event_01,hazard,a young man of 15 years old would have fallen ...
3,Lavenir 2021 - À la recherche d'une jeune fill...,Workflow A,event_01,location,in the Ourthe in Marcourt
4,AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLU...,Workflow B,event_04,impact,loss of human life occurred in the provinces o...
5,AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLU...,Workflow B,event_05,impact,loss of human life occurred in the provinces o...
6,AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLU...,Workflow B,event_25,hazard,"Between 6:0 0 and 18:00 hours, 245.6 mm accumu..."
7,AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLU...,Workflow B,event_29,hazard,At 09:41 hours the north coast and the north i...
8,AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLU...,Workflow B,event_29,time,At 09:41 hours the north coast and the north i...
9,AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLU...,Workflow B,event_31,hazard,"In the region of Almanzora Valley, precipitati..."


Saved unsupported evidence cases to: C:\Users\onssr\Desktop\AIHWS\text2hazard_project\outputs\unsupported_evidence_cases.csv
